# PERO OCR — image(s) vers ALTO XML (version simple)

Tu fournis juste une ou plusieurs images (pas besoin d'ALTO/manifest/toc existants).
Utilise le script officiel `user_scripts/parse_folder.py` du dépôt
[DCGM/pero-ocr](https://github.com/DCGM/pero-ocr) (plus robuste que du code Python
fait main, car son API interne change selon les versions).

**Structure Drive attendue :**
```
MyDrive/
  pero_test/
    images/        ← dépose ici tes images (jpg/png/jp2...)
      p3.jpg
    model/
      pero_eu_cz_print_newspapers_2022-09-26.zip   ← le zip tel quel, pas besoin de le dézipper
```
Le notebook copie le zip en local sur Colab et le décompresse lui-même (plus rapide
que de faire tourner le modèle directement depuis Drive, et t'évite de dézipper
360 Mo à la main). Le résultat ALTO sort dans `MyDrive/pero_test/alto_output/`.

**Avant de lancer :** active un GPU (`Exécution > Modifier le type d'exécution > GPU`)
— sinon passe `DEVICE = 'cpu'` dans la cellule 3 (plus lent).

In [ ]:
# ── Cellule 1 : Dépendances ────────────────────────────────────────────────
# On clone le dépôt pour avoir user_scripts/parse_folder.py (le simple `pip install
# pero-ocr` n'inclut pas les scripts utilisateurs, seulement la librairie).
!git clone -q --depth 1 https://github.com/DCGM/pero-ocr.git /content/pero-ocr-repo
!pip install -q -r /content/pero-ocr-repo/requirements.txt
!pip install -q -e /content/pero-ocr-repo

import torch
print('CUDA disponible :', torch.cuda.is_available())
if not torch.cuda.is_available():
    print("⚠ Pas de GPU détecté — Exécution > Modifier le type d'exécution > GPU "
          "(ou passe DEVICE='cpu' plus bas, mais ce sera lent)")


In [ ]:
# ── Cellule 2 : Montage Drive ──────────────────────────────────────────────
import os, shutil

mountpoint = '/content/gdrive'
if os.path.isdir(mountpoint):
    os.system(f'fusermount -uz {mountpoint} 2>/dev/null || umount -l {mountpoint} 2>/dev/null')
    shutil.rmtree(mountpoint, ignore_errors=True)

from google.colab import drive
drive.mount(mountpoint)
print('Drive monté sur', mountpoint)


In [ ]:
# ── Cellule 3 : Configuration + préparation du modèle (copie locale + dézip) ──
import os, glob, shutil, zipfile

# Dossier contenant tes images (jpg/png/jp2/tif...)
IMAGES_DIR = '/content/gdrive/MyDrive/pero_test/images'      # ← dépose tes images ici

# Dossier de sortie pour les ALTO XML générés
OUTPUT_DIR = '/content/gdrive/MyDrive/pero_test/alto_output'

# Où chercher le modèle sur Drive : soit un .zip, soit un dossier déjà dézippé,
# n'importe où sous pero_test/ (on cherche large pour ne pas dépendre du nom exact).
DRIVE_MODEL_SEARCH_ROOT = '/content/gdrive/MyDrive/pero_test'

DEVICE = 'gpu'   # 'gpu' ou 'cpu'
LOCAL_MODEL_DIR = '/content/pero_model'   # copie locale Colab (plus rapide que lire depuis Drive)

os.makedirs(IMAGES_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(LOCAL_MODEL_DIR, exist_ok=True)

config_name = 'config.ini' if DEVICE == 'gpu' else 'config_cpu.ini'

# 1. Modèle déjà dézippé en local (run précédent dans cette session) ?
local_configs = glob.glob(f'{LOCAL_MODEL_DIR}/**/{config_name}', recursive=True)

if not local_configs:
    # 2. Modèle déjà dézippé sur Drive ?
    drive_configs = glob.glob(f'{DRIVE_MODEL_SEARCH_ROOT}/**/{config_name}', recursive=True)
    if drive_configs:
        MODEL_CONFIG = drive_configs[0]
        print(f'Modèle dézippé trouvé directement sur Drive : {MODEL_CONFIG}')
    else:
        # 3. Sinon on cherche un .zip sur Drive, on le copie en local et on dézippe.
        drive_zips = glob.glob(f'{DRIVE_MODEL_SEARCH_ROOT}/**/*.zip', recursive=True)
        if not drive_zips:
            raise FileNotFoundError(
                f"Aucun modèle trouvé (ni {config_name}, ni .zip) sous {DRIVE_MODEL_SEARCH_ROOT} "
                "— vérifie que tu as bien uploadé le zip du modèle dans MyDrive/pero_test/model/."
            )
        zip_path = drive_zips[0]
        print(f'📦 Zip trouvé sur Drive : {zip_path}')
        local_zip = f'{LOCAL_MODEL_DIR}/model.zip'
        print('📋 Copie locale (plus rapide que de lire depuis Drive à chaque appel)…')
        shutil.copy2(zip_path, local_zip)
        print('📦 Décompression…')
        with zipfile.ZipFile(local_zip, 'r') as zf:
            zf.extractall(LOCAL_MODEL_DIR)
        os.remove(local_zip)
        local_configs = glob.glob(f'{LOCAL_MODEL_DIR}/**/{config_name}', recursive=True)
        if not local_configs:
            raise FileNotFoundError(
                f"{config_name} introuvable après décompression de {zip_path} — "
                f"contenu extrait : {glob.glob(f'{LOCAL_MODEL_DIR}/**/*', recursive=True)[:20]}"
            )
        MODEL_CONFIG = local_configs[0]
else:
    MODEL_CONFIG = local_configs[0]

print(f'Images  : {IMAGES_DIR}')
print(f'Sortie  : {OUTPUT_DIR}')
print(f'Modèle  : {MODEL_CONFIG}')


In [ ]:
# ── Cellule 4 : Lancement de PERO OCR sur le dossier d'images ─────────────
# Utilise le script officiel — gère lui-même détection de mise en page, lignes,
# reconnaissance, et export direct en ALTO XML.
!python3 /content/pero-ocr-repo/user_scripts/parse_folder.py \
    --config "{MODEL_CONFIG}" \
    --input-image-path "{IMAGES_DIR}" \
    --output-alto-path "{OUTPUT_DIR}" \
    --device "{DEVICE}" \
    --skip-processed


In [ ]:
# ── Cellule 5 : Aperçu du texte reconnu (contrôle rapide) ─────────────────
import xml.etree.ElementTree as ET
from pathlib import Path

NS_ALTO = 'http://www.loc.gov/standards/alto/ns-v2#'  # PERO OCR exporte en ALTO v2

_alto_files = sorted(Path(OUTPUT_DIR).glob('*.xml'))
print(f'{len(_alto_files)} fichier(s) ALTO généré(s)\n')

if _alto_files:
    _first = _alto_files[0]
    tree = ET.parse(_first)
    root = tree.getroot()
    lines = []
    for tl in root.iter(f'{{{NS_ALTO}}}TextLine'):
        strings = tl.findall(f'{{{NS_ALTO}}}String')
        text = ' '.join(s.get('CONTENT', '') for s in strings)
        wc_scores = [float(s.get('WC', 0)) for s in strings if s.get('WC')]
        avg_wc = sum(wc_scores) / len(wc_scores) if wc_scores else 0
        lines.append((text, avg_wc))
    print(f'{_first.name} — {len(lines)} lignes (WC = confiance moyenne des mots, 0-1)\n')
    for text, wc in lines[:20]:
        if text.strip():
            print(f'  [{wc:.2f}]', text)
else:
    print('Aucun ALTO généré — vérifie les logs de la cellule 4.')
